# Steering Vectors

# Steering Vectors

## What This Is
Steering vectors (representation engineering, Turner et al., 2023) allow behavioral control of language models by adding a direction in activation space during the forward pass. The key insight: concepts are linearly represented in the residual stream, so adding a vector corresponding to concept C shifts model behavior toward or away from C.

**Finding steering vectors**:
1. Collect contrastive pairs: (positive_prompt, negative_prompt) for the target concept
2. Run both through the model, record activations at a specific layer
3. Compute mean difference vector: steering_vector = mean(positive_acts) - mean(negative_acts)

**Applying steering vectors**: At inference time, add α * steering_vector to the residual stream at the target layer.

In [ ]:
import numpy as np
from typing import List, Tuple, Dict

np.random.seed(42)

D_MODEL = 32
VOCAB = 60
SEQ = 10

class SteeringVectorDemo:
    def __init__(self):
        np.random.seed(3)
        self.W_emb = np.random.randn(VOCAB, D_MODEL) * 0.2
        self.W_pos = np.random.randn(SEQ, D_MODEL) * 0.1
        # Two-layer transformer
        self.W_attn = [np.random.randn(D_MODEL, D_MODEL) * 0.1 for _ in range(2)]
        self.W_ff = [np.random.randn(D_MODEL, 2*D_MODEL) * 0.1 for _ in range(2)]
        self.W_ff_out = [np.random.randn(2*D_MODEL, D_MODEL) * 0.1 for _ in range(2)]
        # Encode a concept in activation space
        # 'positive' concept tokens: 10-20; 'negative': 30-40
        self.concept_direction = np.random.randn(D_MODEL)
        self.concept_direction /= np.linalg.norm(self.concept_direction)
        # Bias W_ff layer 1 to activate the concept for positive tokens
        for tok in range(10, 20):
            self.W_emb[tok] += 0.5 * self.concept_direction
        for tok in range(30, 40):
            self.W_emb[tok] -= 0.5 * self.concept_direction
        self.W_unembed = self.W_emb.T * 0.3
    
    def forward(self, tokens, steering_vec=None, steer_layer=1, steer_coef=0.0):
        T = len(tokens)
        x = self.W_emb[tokens] + self.W_pos[:T]
        layer_acts = [x.copy()]
        
        for l in range(2):
            x = x + 0.3 * (x @ self.W_attn[l])
            x = x + 0.3 * np.maximum(0, x @ self.W_ff[l]) @ self.W_ff_out[l]
            if steering_vec is not None and l == steer_layer:
                x = x + steer_coef * steering_vec[None, :]
            layer_acts.append(x.copy())
        
        return x @ self.W_unembed, layer_acts
    
    def extract_steering_vector(self, positive_tokens, negative_tokens, layer=0):
        pos_acts = np.array([
            self.forward([t])[1][layer+1][0]
            for t in positive_tokens
        ])
        neg_acts = np.array([
            self.forward([t])[1][layer+1][0]
            for t in negative_tokens
        ])
        vec = pos_acts.mean(0) - neg_acts.mean(0)
        return vec / (np.linalg.norm(vec) + 1e-8)

model = SteeringVectorDemo()

# Extract steering vector for the 'positive' concept
sv = model.extract_steering_vector(
    positive_tokens=list(range(10, 20)),
    negative_tokens=list(range(30, 40)),
)

# Test alignment with ground truth concept direction
alignment = float(np.dot(sv, model.concept_direction))
print(f'Steering vector alignment with true concept direction: {alignment:.3f}')
print('(1.0 = perfect recovery, 0 = orthogonal)')

# Apply steering to a neutral input and observe behavioral shift
neutral_tokens = [5, 6, 7, 8]  # neutral tokens (not positive or negative)

logits_base, _ = model.forward(neutral_tokens)
logits_steer_pos, _ = model.forward(neutral_tokens, sv, steer_coef=3.0)
logits_steer_neg, _ = model.forward(neutral_tokens, sv, steer_coef=-3.0)

# Measure how concept-positive tokens are favored
pos_logit_base = logits_base[-1, 10:20].mean()
neg_logit_base = logits_base[-1, 30:40].mean()
pos_logit_steer = logits_steer_pos[-1, 10:20].mean()
neg_logit_steer_neg = logits_steer_neg[-1, 30:40].mean()

print('\nSteering Effect on Next-Token Logits:')
print(f'  Baseline: positive tokens avg logit={pos_logit_base:.3f}, negative={neg_logit_base:.3f}')
print(f'  +3.0 steering: positive avg={logits_steer_pos[-1, 10:20].mean():.3f}')
print(f'  -3.0 steering: negative avg={logits_steer_neg[-1, 30:40].mean():.3f}')
